In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_excel("Online Retail.xlsx")

In [ ]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [ ]:
df.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [7]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


## Data Preprocessing and Cleaning

In [ ]:
# Remove missing descriptions and customer IDs
df = df.dropna(subset=["Description", "CustomerID"])

# Reomve called invoices
df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]

# Remove returns/invalid quantities
df = df[df['Quantity'] > 0]

# Remove invalid prices
df = df[df['UnitPrice'] > 0]


In [9]:
# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

In [10]:
# Change InvoiceDate to datetime
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)

datetime64[ns]


## Feature Engineering (The "Past" Window)

In [ ]:
## RFM Analysis

# Recency: How recently a customer has made a purchase
# Frequency: How often a customer makes a purchase
# Monetary Value: How much money a customer spends on purchases

In [11]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'Revenue': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'MonetaryValue']

In [12]:
rfm.head()

,Recency,Frequency,MonetaryValue
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [ ]:
# Analytic report


# Customer 12347 appears to be the most valuable customer among the sample because they purchased recently (Recency = 2), 
# made multiple purchases (Frequency = 7), and generated substantial revenue. 

# In contrast, Customer 12346 generated the highest historical revenue but only made a single purchase and has been inactive for 326 days, 
# suggesting a high-value customer who may have churned. 

# Customers 12348 and 12349 show future potential, 
# while Customer 12350 exhibits characteristics of a lost customer due to low spending, low purchase frequency, and long inactivity.

In [14]:
## AOV = Total Revenue / Total Orders

customer_metrics = df.groupby('CustomerID').agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique')
)

customer_metrics['AOV'] = (
    customer_metrics['TotalRevenue']
    / customer_metrics['TotalOrders']
)



In [16]:
## Customer Tenure : How long the customer has been with the business.
# Tenure = Last Purchase Date - First Purchase Date

tenure = df.groupby('CustomerID').agg(
    FirstPurchaseDate=('InvoiceDate', 'min'),
    LastPurchaseDate=('InvoiceDate', 'max')
)

tenure['TenureDays'] = (
    tenure['LastPurchaseDate']
    - tenure['FirstPurchaseDate']
).dt.days

##Longer tenure often indicates stronger customer relationships.

In [17]:
## Purchase Gap : Average number of days between purchases.
def avg_purchase_gap(group):
    dates = sorted(group['InvoiceDate'].dt.date.unique())
    if len(dates) < 2:
        return None
    
    gaps = [
        (dates[i] - dates[i-1]).days
        
        for i in range(1, len(dates))
        
        ]
    return sum(gaps) / len(gaps)

In [18]:
purchase_gap = (
    df.groupby('CustomerID')
      .apply(avg_purchase_gap)
      .rename('AvgPurchaseGap')
)

C:\Users\merom\AppData\Local\Temp\ipykernel_15356\1101863591.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(avg_purchase_gap)


In [19]:
snapshot_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)

customer_features = df.groupby('CustomerID').agg(
    Recency=('InvoiceDate',
             lambda x: (snapshot_date - x.max()).days),

    Frequency=('InvoiceNo', 'nunique'),

    MonetaryValue=('Revenue', 'sum'),

    FirstPurchase=('InvoiceDate', 'min'),

    LastPurchase=('InvoiceDate', 'max')
)

customer_features['AOV'] = (
    customer_features['MonetaryValue']
    / customer_features['Frequency']
)

customer_features['TenureDays'] = (
    customer_features['LastPurchase']
    - customer_features['FirstPurchase']
).dt.days

In [ ]:
customer_features = customer_features.join(purchase_gap)

In [22]:
customer_features.head()

,Recency,Frequency,MonetaryValue,FirstPurchase,LastPurchase,AOV,TenureDays,AvgPurchaseGap
CustomerID,,,,,,,,
12346.0,326,1,77183.60,2011-01-18 10:01:00,2011-01-18 10:01:00,77183.600000,0,NaN
12347.0,2,7,4310.00,2010-12-07 14:57:00,2011-12-07 15:52:00,615.714286,365,60.833333
12348.0,75,4,1797.24,2010-12-16 19:09:00,2011-09-25 13:13:00,449.310000,282,94.333333
12349.0,19,1,1757.55,2011-11-21 09:51:00,2011-11-21 09:51:00,1757.550000,0,NaN
12350.0,310,1,334.40,2011-02-02 16:01:00,2011-02-02 16:01:00,334.400000,0,NaN
